# PII Detection with GuardEx

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atliq/guardex-ai/blob/main/docs/notebooks/02_pii_detection.ipynb)

Detect, mask, block, and reversibly tokenize personal data - in-process, no API keys.

You will:

1. Scan text against the 31 built-in entity types
2. Choose between mask and block
3. Tune the confidence threshold
4. Add deny lists and custom regex entities
5. Round-trip PII through the reversible vault

In [ ]:
%pip install -q "guardex-ai[local]"

In [ ]:
from guardex import Guard, DEFAULT_PII_ENTITIES

print(f"{len(DEFAULT_PII_ENTITIES)} built-in entity types:")
print(", ".join(DEFAULT_PII_ENTITIES))

guard = Guard()  # first construction downloads models (~250 MB, cached)

The 31 types cover five groups: personal info (email, phone, SSN, passport, ...), credentials and secrets (password, private key, JWT, ...), API keys and tokens (AWS, GitHub, Stripe, OpenAI, ...), financial (credit card, bank account, IBAN), and network (IP, MAC, hostname, database URL). Detection combines a GLiNER model with regex validators.

## Detect entities

In [ ]:
text = (
    "New ticket from Jane Doe <jane.doe@example.com>, phone 555-867-5309. "
    "Card on file: 4111 1111 1111 1111. Origin host 10.0.0.15."
)
result = guard.screen(text, gate="input")

print(result.text)
print()
for e in result.pii.entities:
    print(f"{e.label:<14} {e.text!r}  score={e.score:.2f}  span=({e.start},{e.end})")

## Mask vs block

`pii_action="mask"` (the default) replaces each entity with a placeholder and lets the request continue - `result.text` carries the masked version. `pii_action="block"` refuses the request instead; use it on surfaces where personal data must never transit.

In [ ]:
from guardex import GuardExPolicy

blocker = Guard(policy=GuardExPolicy(pii_action="block"))
result = blocker.screen("My SSN is 856-45-6789.", gate="input")

print(f"action:  {result.action}")
print(f"blocked: {result.blocked}")

## Thresholds and the allow list

The default threshold is **0.85**. Real PII - emails, SSNs, cards, phone numbers - consistently scores at or above 0.95, while the 0.6-0.8 band is where GLiNER's false positives land (short conversational tokens like "hi" or "ok"). A 21-token allow list ships enabled so those never surface.

The allow list is why casual chat stays clean even at lower thresholds. Watch what happens to the word "hi" when both protections come off:

In [ ]:
loose = Guard(policy=GuardExPolicy(pii_threshold=0.5, pii_allow_list=[]))

for label, g_ in (("default (0.85 + allow list)", guard), ("0.50, allow list off", loose)):
    r = g_.screen("hi", gate="input")
    print(f"{label:<28}: {[(e.label, e.text) for e in r.pii.entities]}")

## Deny lists and custom entities

Three extension points, all on the policy: `pii_deny_list` (exact strings always tagged, score 1.0), `pii_custom_regex` (label to pattern), and `pii_custom_context_keywords` (boost confidence when keywords appear near a match).

In [ ]:
policy = GuardExPolicy(
    pii_deny_list=["PROJ-EAGLE"],
    pii_custom_regex={"employee_id": r"EMP-\d{5}"},
)
custom = Guard(policy=policy)

r = custom.screen("Ticket from EMP-40213 about the PROJ-EAGLE rollout.", gate="input")
print(r.text)
for e in r.pii.entities:
    print(f"{e.label:<14} {e.text!r}  score={e.score:.2f}")

## The PII Vault - reversible tokenization

Masking is one-way. When the model must echo personal data back ("send the confirmation to ..."), vault the entities instead: the LLM sees stable `{{pii:...}}` tokens, and you restore the originals before showing the user. The model never sees the raw values.

In [ ]:
from guardex import PIIVault

vault = PIIVault()
text = "Send the invoice to jane.doe@example.com."

r = guard.screen(text, gate="input")
vaulted, _ = vault.vault_text(text, r.pii)  # populates `vault` in place
print(f"LLM sees:    {vaulted}")

llm_reply = vaulted.replace("Send the invoice to", "Done - invoice sent to")
print(f"LLM replies: {llm_reply}")
print(f"User sees:   {vault.restore(llm_reply)}")

## Next steps

- [03 - Content safety](./03_content_safety.ipynb)
- [PII detection guide](../guides/pii-detection.md) and [PII vault guide](../guides/pii-vault.md)
- Streaming: `Guard.astream(..., vault=vault, restore_mode="buffered")` restores tokens correctly across chunk boundaries